# 📘 Revive an Old Scanned Book — Google Colab

Turn a **scanned (image-only) book PDF** — e.g. Avner's *Introduction to Physical Metallurgy* — into a **cleaned, fully searchable PDF** and extract its **figures**, all in your browser. Nothing to install on your PC.

**How to use:**
1. Runtime → *Run all* (or run each cell top to bottom with ▶️).
2. When prompted in the **Upload** cell, pick your book PDF from your computer (drag it from your OneDrive/Courses folder).
3. Wait for OCR to finish (a big textbook can take a while).
4. The searchable PDF and a `figures.zip` download automatically at the end.

> ⚠️ **Metallurgy note:** colorizing works well for *diagrams* (iron–carbon, TTT curves) but invents scientifically **false** colors on etched *micrographs*. Keep reference micrographs grayscale. This is for reviving **your own copy** for personal study.

### 1. Install the OCR tools (~1 min)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr ghostscript poppler-utils unpaper pngquant > /dev/null
!pip -q install ocrmypdf
print('✅ Tools installed')

### 2. Upload your scanned book PDF
Run this cell, then choose your PDF. (For a large file you can instead mount Google Drive — see the optional cell at the bottom.)

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
INPUT_PDF = next(iter(uploaded))
print(f'✅ Uploaded: {INPUT_PDF}  ({os.path.getsize(INPUT_PDF)//1024//1024} MB)')

### 3. Clean + OCR the whole book (makes it searchable)
Keeps every figure exactly as-is and adds an invisible searchable text layer. `--skip-text` makes it safe to re-run.

In [ ]:
OUTPUT_PDF = os.path.splitext(INPUT_PDF)[0] + '_searchable.pdf'

!ocrmypdf --language eng --deskew --rotate-pages --clean --optimize 1 \
          --skip-text "$INPUT_PDF" "$OUTPUT_PDF"

print('✅ Searchable book written:', OUTPUT_PDF)

### 4. Extract the figures
`fig-*.png` = the raw embedded images (best for micrographs). `page-*.png` = full-page renders (best for cropping diagrams).

In [ ]:
import os, glob
os.makedirs('figures', exist_ok=True)
os.makedirs('pages', exist_ok=True)

!pdfimages -png -p "$OUTPUT_PDF" figures/fig
!pdftoppm  -png -r 200 "$OUTPUT_PDF" pages/page

print('✅ Embedded figures:', len(glob.glob('figures/*.png')))
print('✅ Page renders   :', len(glob.glob('pages/*.png')))

### 5. Download the results
Downloads the searchable PDF and a zip of the extracted figures.

In [ ]:
from google.colab import files
!zip -q -r figures.zip figures pages
files.download(OUTPUT_PDF)
files.download('figures.zip')
print('✅ Done — check your browser downloads')

---
### (Optional) Colorize figures
Use a hosted colorizer on the diagrams you extracted — **Palette.fm** (prompt-controllable, great for diagrams) or **MyHeritage**. Upload the PNGs from `figures.zip`.

Remember: recolor **diagrams** freely; keep **micrographs** grayscale (AI colors on etched microstructures are decorative, not accurate).

---
### (Optional) Use a huge file straight from Google Drive
Instead of uploading, mount your Drive and point `INPUT_PDF` at the file:
```python
from google.colab import drive
drive.mount('/content/drive')
INPUT_PDF = '/content/drive/MyDrive/Introduction to Physical Metallurgy (1974).pdf'
```
Then re-run cells 3–5.